# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wajiha-Waqar/FlyRankInternship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

My lane is content-refresh prioritization: identifying pages that may deserve review based on signals available before the outcome occurs.

For this first modeling experiment, I use Logistic Regression and Random Forest.

Logistic Regression is the simple, interpretable model. It provides a probability that can be used to rank pages and gives a useful benchmark for whether a learned linear relationship can improve on my Week-4 rule.

Random Forest is the tree-based model. I chose it because content performance can depend on nonlinear relationships between visibility, CTR, search position, engagement, and traffic sources. It provides a second model that can capture interactions that Logistic Regression may miss.

I will evaluate the models as ranking systems rather than judging them only by a classification threshold. This matches the practical question of my lane: which pages should be reviewed first?

The models will be compared with my frozen Week-4 baseline using the same test set and the same ranking metrics. I will report Precision@K and Average Precision, together with the positive-class base rate.

I will keep the models deliberately simple. The purpose of this experiment is not to maximize complexity, but to determine whether a learned model provides useful improvement over the transparent rule baseline.


In [1]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

con.sql(f"""
CREATE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [2]:
con.execute("DROP TABLE IF EXISTS fact_content_daily_performance")

In [3]:
march_df = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

april_df = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

print("March shape:", march_df.shape)
print("April shape:", april_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March shape: (3611061, 31)
April shape: (3901060, 31)


In [4]:
def aggregate_month(df):
    df = df.copy()

    df["gsc_impressions"] = df["gsc_impressions"].fillna(0)
    df["gsc_clicks"] = df["gsc_clicks"].fillna(0)

    result = (
        df.groupby(
            ["client_hash_id", "content_hash_id"],
            as_index=False
        )
        .agg(
            gsc_impressions=("gsc_impressions", "sum"),
            gsc_clicks=("gsc_clicks", "sum"),
            gsc_avg_position=("gsc_avg_position", "mean"),
            ga4_pageviews=("ga4_pageviews", "sum"),
            ga4_sessions=("ga4_sessions", "sum"),
            ga4_users=("ga4_users", "sum"),
            ga4_engaged_sessions=("ga4_engaged_sessions", "sum"),
            ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum"),
            sessions_organic=("sessions_organic", "sum"),
            sessions_direct=("sessions_direct", "sum"),
            sessions_referral=("sessions_referral", "sum"),
            sessions_social=("sessions_social", "sum"),
            sessions_paid=("sessions_paid", "sum"),
            sessions_ai=("sessions_ai", "sum"),
            ai_chatgpt=("ai_chatgpt", "sum"),
            ai_perplexity=("ai_perplexity", "sum"),
            ai_gemini=("ai_gemini", "sum"),
            ai_copilot=("ai_copilot", "sum"),
            ai_claude=("ai_claude", "sum"),
            ai_meta=("ai_meta", "sum"),
            ai_other=("ai_other", "sum"),
            scroll_events=("scroll_events", "sum")
        )
    )

    result["ctr"] = np.where(
        result["gsc_impressions"] > 0,
        result["gsc_clicks"] / result["gsc_impressions"],
        np.nan
    )

    return result

In [5]:
march_month = aggregate_month(march_df)
april_month = aggregate_month(april_df)

In [6]:
panel = march_month.merge(
    april_month[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    suffixes=("_march", "_april")
)

In [7]:
panel["ctr_change_pct"] = (
    (panel["ctr_april"] - panel["ctr_march"])
    / panel["ctr_march"]
)

In [8]:
panel["refresh_outcome"] = (
    (panel["gsc_impressions_march"] >= 100)
    &
    (panel["ctr_march"] > 0)
    &
    (panel["ctr_change_pct"] <= -0.20)
).astype(int)

Interpretation:

A page is considered a positive future outcome when it had at least 100 March impressions, had measurable March CTR, and its April CTR was at least 20% lower than its March CTR.

In [9]:
panel["refresh_outcome"].value_counts()

,count
refresh_outcome,
0,120915
1,37634


In [10]:
panel["refresh_outcome"].mean()

np.float64(0.2373651047941015)

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a grouped client-level train/test split rather than randomly splitting individual content pages.

The reason is that multiple pages from the same client can share characteristics, traffic patterns, and measurement conditions. Randomly placing pages from the same client into both training and test sets could therefore make the test result look better than it would be on genuinely unseen clients.

I hold out complete clients for testing and use the remaining clients for training. The March data is used as the feature window, while the April data is used only to construct the future outcome label.

This prevents the model from using future April performance as a feature.

I fix the random seed so that the split and model results are reproducible.


In [14]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        panel,
        panel["refresh_outcome"],
        groups=panel["client_hash_id"]
    )
)

train = panel.iloc[train_idx].copy()
test = panel.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", train["client_hash_id"].nunique())
print("Test clients:", test["client_hash_id"].nunique())

Train rows: 137445
Test rows: 21104
Train clients: 36
Test clients: 10


In [15]:
assert set(train["client_hash_id"]).isdisjoint(
    set(test["client_hash_id"])
)

print("No client overlap.")

No client overlap.


In [21]:
feature_cols = [
    "gsc_impressions_march",
    "gsc_clicks_march",
    "ctr_march",
]

In [22]:
[c for c in feature_cols if c not in panel.columns]

[]

In [23]:
X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()

y_train = train["refresh_outcome"]
y_test = test["refresh_outcome"]

In [24]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

logistic.fit(X_train, y_train)

logistic_scores = logistic.predict_proba(X_test)[:, 1]

In [26]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf.fit(X_train, y_train)

rf_scores = rf.predict_proba(X_test)[:, 1]

In [28]:
test["baseline_score"] = (
    (test["gsc_impressions_march"] >= 500).astype(int)
    +
    (test["ctr_march"] < 0.01).astype(int)
)

In [29]:
from sklearn.metrics import average_precision_score

In [30]:
ap_baseline = average_precision_score(
    y_test,
    test["baseline_score"]
)

ap_logistic = average_precision_score(
    y_test,
    logistic_scores
)

ap_rf = average_precision_score(
    y_test,
    rf_scores
)

In [31]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[order].mean()

In [32]:
for k in [20, 50, 100]:
    print(
        "K =", k,
        "Baseline =", precision_at_k(
            y_test,
            test["baseline_score"],
            k
        ),
        "Logistic =", precision_at_k(
            y_test,
            logistic_scores,
            k
        ),
        "Random Forest =", precision_at_k(
            y_test,
            rf_scores,
            k
        )
    )

K = 20 Baseline = 0.3 Logistic = 0.75 Random Forest = 0.85
K = 50 Baseline = 0.4 Logistic = 0.62 Random Forest = 0.8
K = 100 Baseline = 0.5 Logistic = 0.66 Random Forest = 0.84


In [33]:
base_rate = y_test.mean()

print("Test base rate:", base_rate)

Test base rate: 0.35718347232752085


In [34]:
results = []

for k in [20, 50, 100]:
    results.append({
        "model": "Baseline",
        "precision_at_k": k,
        "precision": precision_at_k(
            y_test,
            test["baseline_score"],
            k
        ),
        "average_precision": ap_baseline,
        "base_rate": base_rate
    })

    results.append({
        "model": "Logistic Regression",
        "precision_at_k": k,
        "precision": precision_at_k(
            y_test,
            logistic_scores,
            k
        ),
        "average_precision": ap_logistic,
        "base_rate": base_rate
    })

    results.append({
        "model": "Random Forest",
        "precision_at_k": k,
        "precision": precision_at_k(
            y_test,
            rf_scores,
            k
        ),
        "average_precision": ap_rf,
        "base_rate": base_rate
    })

comparison = pd.DataFrame(results)

comparison

,model,precision_at_k,precision,average_precision,base_rate
0,Baseline,20,0.30,0.497366,0.357183
1,Logistic Regression,20,0.75,0.664095,0.357183
2,Random Forest,20,0.85,0.770925,0.357183
3,Baseline,50,0.40,0.497366,0.357183
4,Logistic Regression,50,0.62,0.664095,0.357183
5,Random Forest,50,0.80,0.770925,0.357183
6,Baseline,100,0.50,0.497366,0.357183
7,Logistic Regression,100,0.66,0.664095,0.357183
8,Random Forest,100,0.84,0.770925,0.357183


## 3. Train + compare vs my baseline

I evaluated the frozen Week-4 baseline, Logistic Regression, and Random Forest on the same client-held-out test set.

The primary evaluation is ranking-based because the practical use case is to decide which pages should be reviewed first. I therefore report Precision@K at multiple review capacities and Average Precision. The test-set positive-class base rate is also reported so that the ranking results have context.

The baseline was not retrained or optimized for the test set. It uses the Week-4 rule: one point for high visibility and one point for low CTR.

The learned models use only March features, while the April outcome is used only as the evaluation label.

|index|model|precision\_at\_k|precision|average\_precision|base\_rate|
|---|---|---|---|---|---|
|0|Baseline|20|0\.3|0\.49736597687918116|0\.35718347232752085|
|1|Logistic Regression|20|0\.75|0\.664095259006337|0\.35718347232752085|
|2|Random Forest|20|0\.85|0\.7709248445885697|0\.35718347232752085|
|3|Baseline|50|0\.4|0\.49736597687918116|0\.35718347232752085|
|4|Logistic Regression|50|0\.62|0\.664095259006337|0\.35718347232752085|
|5|Random Forest|50|0\.8|0\.7709248445885697|0\.35718347232752085|
|6|Baseline|100|0\.5|0\.49736597687918116|0\.35718347232752085|
|7|Logistic Regression|100|0\.66|0\.664095259006337|0\.35718347232752085|
|8|Random Forest|100|0\.84|0\.7709248445885697|0\.35718347232752085|

The results show whether the learned models provide useful ranking improvement over the simple rule. I will not treat a model as better merely because it is more complex. The important question is whether it improves the ranking of pages that later meet the defined outcome, especially at the review capacity that would be practical for an SEO team.


In [35]:
rf_model = rf.named_steps["model"]

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance.head(10)

,feature,importance
2,ctr_march,0.402722
1,gsc_clicks_march,0.378393
0,gsc_impressions_march,0.218885


In [36]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean
}).sort_values(
    "importance_mean",
    ascending=False
)

perm_df.head(10)

,feature,importance_mean
2,ctr_march,0.102454
1,gsc_clicks_march,0.069547
0,gsc_impressions_march,0.047766


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [38]:
test_results = test[
    [
        "client_hash_id",
        "content_hash_id",
        "refresh_outcome",
        "gsc_impressions_march",
        "ctr_march",
        "baseline_score"
    ]
].copy()

test_results["logistic_score"] = logistic_scores
test_results["rf_score"] = rf_scores

In [39]:
test_results["baseline_rank"] = (
    test_results["baseline_score"]
    .rank(method="first", ascending=False)
)

test_results["rf_rank"] = (
    test_results["rf_score"]
    .rank(method="first", ascending=False)
)

disagreements = test_results[
    test_results["baseline_score"] !=
    (test_results["rf_score"] >= test_results["rf_score"].median()).astype(int)
].copy()

In [40]:
test_results["baseline_rank"] = (
    test_results["baseline_score"]
    .rank(method="first", ascending=False)
)

test_results["rf_rank"] = (
    test_results["rf_score"]
    .rank(method="first", ascending=False)
)

test_results["rank_difference"] = (
    test_results["baseline_rank"] -
    test_results["rf_rank"]
)

In [41]:
test_results.sort_values(
    "rank_difference",
    ascending=False
).head(10)

,client_hash_id,content_hash_id,refresh_outcome,gsc_impressions_march,ctr_march,baseline_score,logistic_score,rf_score,baseline_rank,rf_rank,rank_difference
152384,client_fef1a8f436438636,content_98fac9164cb33b7e,1,100,0.010000,0,0.199373,0.803163,21062.0,155.0,20907.0
150031,client_fef1a8f436438636,content_550a1ae58c77f8d2,1,100,0.010000,0,0.199373,0.803163,21026.0,154.0,20872.0
144124,client_e5c2aa26a8598242,content_6274d20a30484158,1,149,0.020134,0,0.204098,0.781437,20908.0,252.0,20656.0
151143,client_fef1a8f436438636,content_755aa80f383911bc,1,139,0.021583,0,0.204266,0.775261,21045.0,410.0,20635.0
155650,client_fef1a8f436438636,content_f744c735415c4bcc,1,131,0.015267,0,0.201805,0.772970,21103.0,488.0,20615.0
152049,client_fef1a8f436438636,content_8efdfeb9abf2fe72,1,124,0.024194,0,0.204592,0.774319,21058.0,453.0,20605.0
154020,client_fef1a8f436438636,content_c955e74cb32e0a33,1,132,0.015152,0,0.201793,0.771866,21084.0,493.0,20591.0
151341,client_fef1a8f436438636,content_7b5cdf12e764ac88,1,140,0.014286,0,0.201708,0.773719,21048.0,480.0,20568.0
155888,client_fef1a8f436438636,content_fe812e92ab63811f,1,124,0.016129,0,0.201897,0.769775,21104.0,539.0,20565.0
149902,client_fef1a8f436438636,content_51be3de520a2f3cc,1,138,0.014493,0,0.201728,0.774257,21019.0,457.0,20562.0


In [43]:
disagreement_cases = test_results[
    [
        "content_hash_id",
        "refresh_outcome",
        "gsc_impressions_march",
        "ctr_march",
        "baseline_score",
        "logistic_score",
        "rf_score",
        "rank_difference"
    ]
].sort_values(
    "rank_difference",
    ascending=False
).head(3)

disagreement_cases

,content_hash_id,refresh_outcome,gsc_impressions_march,ctr_march,baseline_score,logistic_score,rf_score,rank_difference
152384,content_98fac9164cb33b7e,1,100,0.010000,0,0.199373,0.803163,20907.0
150031,content_550a1ae58c77f8d2,1,100,0.010000,0,0.199373,0.803163,20872.0
144124,content_6274d20a30484158,1,149,0.020134,0,0.204098,0.781437,20656.0


## 4. Errors and interpretation

The model and the Week-4 baseline do not rank every page in the same way. This is expected because the baseline uses only two threshold rules, while the learned models can combine multiple signals.

The main differences were examined using the test-set ranking and the future outcome label.

### Feature interpretation

The strongest features according to permutation importance were:

1. **ctr_march** (importance_mean ≈ 0.1025)
2. **gsc_clicks_march** (importance_mean ≈ 0.0695)
3. **gsc_impressions_march** (importance_mean ≈ 0.0478)

This ordering matches the built-in Random Forest `feature_importances_` (ctr_march ≈ 0.403, gsc_clicks_march ≈ 0.378, gsc_impressions_march ≈ 0.219), so the two importance methods agree on ranking even though their scales differ.

These features are plausible because the outcome label is built directly from CTR behavior: a page is only labeled positive if it already had non-zero March CTR that then fell sharply in April. A page's current `ctr_march` value is the feature closest, mechanically, to that future decline, so it is reasonable that it dominates. `gsc_clicks_march` is a close second because it reflects how much real engagement a page is generating in absolute terms, which interacts with CTR to describe how fragile a page's current performance is. `gsc_impressions_march` matters least on its own, since visibility alone does not indicate whether the engagement on that visibility is deteriorating.

I do not interpret feature importance as proof of causation. It only indicates which available features were useful for the model's predictions.

### Disagreement cases

Three concrete disagreement cases were inspected, taken from the pages with the largest `rank_difference` between the baseline and the Random Forest.

**Case 1: content_98fac9164cb33b7e**
The baseline ranked this page near the very bottom of the test set (baseline_rank ≈ 21,062 of 21,104), while the Random Forest ranked it near the top (rf_rank = 155). The page had `gsc_impressions_march = 100` and `ctr_march = 0.01`, which lands just outside both baseline rules — impressions are far below the 500-impression threshold, and CTR is not strictly below 0.01, so `baseline_score = 0`. The actual outcome was **1**: this page's CTR did decline by 20%+ into April. This shows the baseline's fixed thresholds can completely miss pages that are already borderline-weak on CTR even at moderate traffic, while the Random Forest picked up on that borderline pattern.

**Case 2: content_550a1ae58c77f8d2**
The models disagreed for essentially the same observed reason as Case 1: `gsc_impressions_march = 100` and `ctr_march = 0.01`, again just outside both baseline thresholds, giving `baseline_score = 0` and a bottom-of-list baseline rank (≈21,026), while the Random Forest scored it 0.803 and ranked it 154th. This case belongs to the same client (`client_fef1a8f436438636`) as several other top disagreement cases, suggesting the model is repeatedly catching a specific low-CTR, moderate-traffic pattern that recurs across that client's pages rather than a one-off outlier. The actual outcome was **1**.

**Case 3: content_6274d20a30484158**
The model over-ranked this page relative to the baseline: Random Forest score 0.781 (rank 252) versus a baseline rank near 20,908, driven by `gsc_impressions_march = 149` and `ctr_march ≈ 0.020`. Unlike Cases 1–2, this page belongs to a different client (`client_e5c2aa26a8598242`), so the pattern is not confined to a single client. The observed signals suggest the Random Forest is generalizing a combination of "moderate impressions + low-but-not-lowest CTR" as risky, rather than memorizing one client's traffic profile. The actual outcome was **1**.

Across all three cases, the Random Forest identified true positives that the baseline's binary thresholds missed entirely because the underlying values sat just below the rule cutoffs. These examples show why the model score should be treated as decision support rather than an automatic refresh decision.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.